In [1]:
import pandas as pd
import numpy as np
import sklearn
from catboost import CatBoostClassifier
from sklearn.metrics import average_precision_score
from src.preprocessing import train_valid_test_split
from src.evaluate import precision_k, score_precision_10
from src.feat_eng import add_log1p_feat
from sklearn.model_selection import GridSearchCV
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

In [2]:
df = pd.read_csv('../data/bank-full.csv', sep=';')
train, valid, test = train_valid_test_split(df)
k = int(len(valid) * 0.1)

In [3]:
train['pdays_prev_camp'] = train['pdays'].ne(-1).astype('int8')
train['pdays'] = train['pdays'].replace(-1, np.nan)

valid['pdays_prev_camp'] = valid['pdays'].ne(-1).astype('int8')
valid['pdays'] = valid['pdays'].replace(-1, np.nan)

test['pdays_prev_camp'] = test['pdays'].ne(-1).astype('int8')
test['pdays'] = test['pdays'].replace(-1, np.nan)

/var/folders/d5/hmchxs4s0v7bc772rvxtlcdw0000gn/T/ipykernel_85811/2034733883.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train['pdays_prev_camp'] = train['pdays'].ne(-1).astype('int8')
/var/folders/d5/hmchxs4s0v7bc772rvxtlcdw0000gn/T/ipykernel_85811/2034733883.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train['pdays'] = train['pdays'].replace(-1, np.nan)
/var/folders/d5/hmchxs4s0v7bc772rvxtlcdw0000gn/T/ipykernel_85811/2034733883.py:4: SettingWithCopyWarning: 
A value is trying to be set on a 

In [33]:

# cols_to_log = ['campaign', 'previous', 'pdays']
#
# train = add_log1p_feat(train, cols_to_log)
# valid = add_log1p_feat(valid, cols_to_log)
#
# train['avg_balance'] = (
#     train.groupby('job')['balance'].transform('mean')
# )
# valid['avg_balance'] = (
#     valid.groupby('job')['balance'].transform('mean')
# )

# train['job_freq'] = (
#     train.groupby('job')['job'].transform('count') / train.job.count()
# )
# valid['job_freq'] = (
#     valid.groupby('job')['job'].transform('count') / valid.job.count()
# )

# cluster_tr = train.loc[:, ['campaign', 'previous', 'pdays']].dropna()
# print(cluster_tr)
# kmeans = KMeans(n_clusters=5, random_state=0)
# kmeans.fit(cluster_tr)
# train['cluster'] = kmeans.fit_predict(cluster_tr)
# train['cluster'] = train['cluster'].astype('category')
# train.cluster.describe()
#
# cluster_val = valid.loc[:, ['campaign', 'previous', 'pdays']].dropna()
# valid['cluster'] = kmeans.predict(cluster_val)
# valid['cluster'] = valid['cluster'].astype('category')

# train['balance_age_ratio'] = abs(train.balance) / train.age
# valid['balance_age_ratio'] = abs(valid.balance) / valid.age

In [34]:
# train['new_balance'] = np.sign(train['balance']) * np.log1p(abs(train['balance']))
# valid['new_balance'] = np.sign(valid['balance']) * np.log1p(abs(valid['balance']))
#
# scaler = StandardScaler()
# train['new_balance'] = scaler.fit_transform(train[['new_balance']])
# valid['new_balance'] = scaler.transform(valid[['new_balance']])
#
#
# kmeans = KMeans(n_clusters=3, random_state=0)
# cluster_tr = train.loc[:, ['new_balance', 'age']]
# train['cluster'] = kmeans.fit_predict(cluster_tr)
# train['cluster'] = train['cluster'].astype('category')
# train.cluster.describe()
#
# cluster_val = valid.loc[:, ['new_balance', 'age']]
# valid['cluster'] = kmeans.predict(cluster_val)
# valid['cluster'] = valid['cluster'].astype('category')

/var/folders/d5/hmchxs4s0v7bc772rvxtlcdw0000gn/T/ipykernel_77550/2332875064.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train['new_balance'] = np.sign(train['balance']) * np.log1p(abs(train['balance']))
/var/folders/d5/hmchxs4s0v7bc772rvxtlcdw0000gn/T/ipykernel_77550/2332875064.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  valid['new_balance'] = np.sign(valid['balance']) * np.log1p(abs(valid['balance']))
/var/folders/d5/hmchxs4s0v7bc772rvxtlcdw0000gn/T/ipykernel_77550/2332875064.py:5: SettingW

In [4]:
X_train = train.drop(columns=['y', 'duration'])
y_train = train["y"]
X_valid = valid.drop(columns=['y', 'duration'])
y_valid = valid['y']
X_test = test.drop(columns=['y', 'duration'])
y_test = test["y"]

In [5]:

cat_cols = X_train.select_dtypes(
    include=["object", "category", "string", "bool"]
)
cat_indices = [
      i for i, col in enumerate(X_train.columns)
      if X_train[col].dtype.name in ("object", "string", "category")
        ]

# eval_pool = Pool(X_valid, y_valid, cat_features=cat_indices)

In [6]:
model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.01,
    loss_function="Logloss",
    depth=8,
    random_seed=0,
    l2_leaf_reg=3,
)
model.fit(X_train, y_train, cat_features=cat_indices, verbose=100)

0:	learn: 0.6807804	total: 91.4ms	remaining: 1m 31s
100:	learn: 0.2526064	total: 3.32s	remaining: 29.5s
200:	learn: 0.2107703	total: 6.35s	remaining: 25.3s
300:	learn: 0.2018884	total: 9.69s	remaining: 22.5s
400:	learn: 0.1980970	total: 13.3s	remaining: 19.8s
500:	learn: 0.1958630	total: 16.6s	remaining: 16.5s
600:	learn: 0.1941790	total: 20.2s	remaining: 13.4s
700:	learn: 0.1928650	total: 24.5s	remaining: 10.5s
800:	learn: 0.1916276	total: 29.3s	remaining: 7.29s
900:	learn: 0.1903406	total: 33.2s	remaining: 3.65s
999:	learn: 0.1888990	total: 38.3s	remaining: 0us


CatBoostClassifier(depth=8, iterations=1000, l2_leaf_reg=3, learning_rate=0.01, loss_function='Logloss', random_seed=0)

In [38]:
# X_search = pd.concat([X_train, X_valid], ignore_index=True)
# y_search = pd.concat([y_train, y_valid], ignore_index=True)
#
# param_grid = {
#       "depth": [4, 6, 8],
#       "learning_rate": [0.01, 0.03, 0.1],
#       "l2_leaf_reg": [3, 10],
# }
#
# grid = GridSearchCV(estimator=model,
#     param_grid=param_grid,
#     cv=[(np.arange(len(X_train)), np.arange(len(X_train), len(X_search)))],
#     scoring=score_precision_10,
#     refit=False,
#     n_jobs=1,
#     error_score='raise',
# )
#
# grid.fit(X_search, y_search, cat_features=cat_indices, verbose=100)


In [7]:
y_pred = model.predict(X_valid)

In [8]:
yes_idx = list(model.classes_).index("yes")
y_score = model.predict_proba(X_valid)[:, yes_idx]
prec_k = precision_k(y_valid, y_score, k=k)
avg_precision = average_precision_score(y_valid, y_score, pos_label="yes")
# avg_precision = model.eval_metrics(data=eval_pool, metrics='Precision')

In [11]:
y_test_pred = model.predict(X_test)
y_test_score = model.predict_proba(X_test)[:, yes_idx]
prec_k_test = precision_k(y_test, y_score, k=k)
avg_precision_test = average_precision_score(y_test, y_test_score, pos_label="yes")

pd.Series({
    "precision_at_10pct_test": prec_k_test,
    "average_precision_test": avg_precision_test,
})

precision_at_10pct_test    0.278761
average_precision_test     0.466768
dtype: float64

In [41]:
pd.Series({
    "precision_at_10pct": prec_k,
    "average_precision": avg_precision,
})

precision_at_10pct    0.264012
average_precision     0.225392
dtype: float64

#### exp 1:
- depth 6, iteration 500, random seed = 0, logloss, lr = 0.01
- metrics: prec_k = 0.2743, avg_prec = 0.2322

#### exp 2:
- same model pars, add feature pdays_prev_camp
- metrics: prec_k = 0.2935, avg_prec = 0.2361

#### exp 3:
- {'depth': 8, 'l2_leaf_reg': 3, 'learning_rate': 0.01} - best params
- metrics: prec_k = 0.2979, avg_prec = 0.2388

#### exp 4:
- best params, logp1 of 'campaign', 'previous', 'pdays' features
- prec_k = 0.2669, avg_prec = 0.2213
- revert to exp 3

#### exp 5:
- best params, add feature 'avg_balance' by job
- prec_k = 0.2581, avg_prec = 0.2271
- revert to exp 3

#### exp 6:
- best params, add feature 'job_freq'
- precision_at_10pct = 0.2758, average_precision = 0.2340
- revert to exp 3

#### exp 7:
- best params, add feature 'cluster' - 5-means clusterization on 'campaign', 'previous'
- precision_at_10pct = 0.2787, average_precision = 0.2391
- revert to exp 3

#### exp 8
- best params, add feature 'balance_age_ratio'
- precision_at_10pct = 0.2802, average_precision = 0.2336
- revert to exp 3

#### exp 9
- best params, add feature 'cluster' - 3-means clusterization on sign(balance) log1p(balance) and age
- precision_at_10pct = 0.2640, average_precision = 0.225392
- revert to exp 3


#### Final results
precision_at_10pct    0.2640
average_precision     0.2253

exp 3 params.

precision@k baseline 0 = 0.1151
precision@k baseline 1 = 0.2079
precision@k final_model = 0.2640